In [1]:
import os
import joblib
import warnings
os.environ["OMP_NUM_THREADS"] = "1"
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.svm import SVR
from google.colab import files
import matplotlib.pyplot as plt
from keras.optimizers import Adam
from keras.models import Sequential
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.layers import LSTM, Dense, Dropout
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [2]:
# Open a file upload dialog
uploaded = files.upload()

Saving Cleaned Dataset.csv to Cleaned Dataset (2).csv


In [3]:
# Import dataset
dataset = pd.read_csv("Cleaned Dataset.csv")
dataset

,Country,Year,BEV Percentage (Total Number Of Registrations),BEV (New Registrations),GDP,CPI,EG,Recharging Points,AC Recharging Speed (km/h),DC Recharging Speed (km/h),...,Real Range,Purchase price (EUR),Log_BEV Percentage (Total Number Of Registrations),Log_BEV (New Registrations),Log_Recharging Points,Log_GDP,Log_CPI,Log_Available,Log_DC Recharging Speed (km/h),YJ_Purchase price (EUR)
0,Austria,2011,0.18,631.0,308167.0,3.286579,1.6,963,17.500000,175.000000,...,95.000000,30469.93,0.165514,6.448889,6.871091,12.638400,1.455489,1.609438,5.170484,-1.493084
1,Austria,2012,0.13,427.0,316589.4,2.485676,1.0,1063,29.169521,217.394139,...,95.000000,30469.93,0.122218,6.059123,6.969791,12.665364,1.248662,1.609438,5.386301,-1.493084
2,Austria,2013,0.21,656.0,321191.7,2.000156,0.4,1173,48.200000,270.000000,...,142.220000,30469.93,0.190620,6.487684,7.068172,12.679797,1.098664,2.197225,5.602119,-1.493084
3,Austria,2014,0.44,1344.0,330113.5,1.605812,1.0,1393,22.600000,240.000000,...,155.770000,35555.64,0.364643,7.204149,7.239933,12.707195,0.957744,2.639057,5.484797,-0.885657
4,Austria,2015,0.55,1699.0,342083.5,0.896563,0.6,1625,47.700000,381.250000,...,215.220000,42052.00,0.438255,7.438384,7.393878,12.742813,0.640043,3.091042,5.946075,0.277809
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,Sweden,2020,9.55,27897.0,5012855.0,0.497367,-1.3,47680,39.325301,403.164557,...,286.200000,38311.38,2.356126,10.236310,10.772288,15.427516,0.403708,4.584967,6.001822,-0.451036
374,Sweden,2021,19.09,57454.0,5417760.0,2.163197,1.3,70537,40.904348,493.391304,...,308.730000,40745.01,3.000222,10.958757,11.163907,15.505193,1.151583,5.129899,6.203327,0.003737
375,Sweden,2022,32.84,94603.0,5816415.0,8.369291,3.5,83510,49.314815,594.259259,...,345.440000,43874.27,3.521644,11.457455,11.332734,15.576195,2.237437,5.537334,6.388997,0.697159
376,Sweden,2023,38.42,111338.0,6143187.0,8.548625,1.2,126758,70.307692,649.230769,...,361.794521,45999.05,3.674273,11.620335,11.750043,15.630854,2.256397,5.918894,6.477327,1.244323


In [4]:
# Define the target
target = 'Log_BEV Percentage (Total Number Of Registrations)'

In [5]:
# Load the PCA tools
tools = joblib.load("pca_processors.pkl")
scaler = tools['scaler']
pca = tools['pca_model']
features = tools['feature_names']

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.3.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.3.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
# Data splitting into train data and test data
pre_covid_df = dataset[dataset['Year'].between(2020, 2023)].copy()
post_covid_df = dataset[dataset['Year'] >= 2024].copy()

Please refer the jupyter notebook - **3.1.Feature Selection_BEV Percentage (Primary Target)** to understand the imputation

In [7]:
# Check which columns have NaNs and how many
print("Missing Values Per Column on Train data (2020-2023)")
print(pre_covid_df[features].isna().sum())

print("\nMissing Values Per Column Test data (2024)")
print(post_covid_df[features].isna().sum())

Missing Values Per Column on Train data (2020-2023)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           1
dtype: int64

Missing Values Per Column Test data (2024)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           0
dtype: int64


In [8]:
# Checking which rows are empty
print("Missing Log_CPI in Train data (2020-2023)")
print(pre_covid_df[pre_covid_df['Log_CPI'].isna()].reset_index()[['Country', 'Year', 'Log_CPI']])

Missing Log_CPI in Train data (2020-2023)
  Country  Year  Log_CPI
0  Greece  2020      NaN


In [9]:
# Imputation for Log_CPI NaNs by filling the spaces with the median

# Calculate the median Log_CPI for every country of train data
country_medians = pre_covid_df.groupby('Country')['Log_CPI'].median()

# Function to fill the gaps
def impute_by_country(df):
    df_clean = df.copy()

    # Loop through the data, if Log_CPI is missing, we look up that specific country's median from our 'country_medians' list.
    df_clean['Log_CPI'] = df_clean['Log_CPI'].fillna(df_clean['Country'].map(country_medians))

    return df_clean

# Apply the fix to both datasets
pre_covid_fixed = impute_by_country(pre_covid_df)
post_covid_fixed = impute_by_country(post_covid_df)

# Extract 10 features
X_train_imputed = pre_covid_fixed[features]
X_test_imputed = post_covid_fixed[features]

In [10]:
# Verfication
print("Missing Values Per Column on Train data (2020-2023)")
print(pre_covid_fixed[features].isna().sum())

print("\nMissing Values Per Column Test data (2024)")
print(post_covid_fixed[features].isna().sum())

Missing Values Per Column on Train data (2020-2023)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           0
dtype: int64

Missing Values Per Column Test data (2024)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           0
dtype: int64


In [11]:
# Scale the features using the scaler loaded from the .pkl file
X_train_scaled = scaler.transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [12]:
# Transform the features into 3 Principal Components using the loaded PCA model
# and convert them back to a dataframe for easy handling
X_train_pca = pd.DataFrame(pca.transform(X_train_scaled), columns=['PC1', 'PC2'], index=pre_covid_fixed.index)
X_test_pca = pd.DataFrame(pca.transform(X_test_scaled), columns=['PC1', 'PC2'], index=post_covid_fixed.index)

In [13]:
# Define the targets
y_train = pre_covid_fixed[target]
y_test = post_covid_fixed[target]

print(f"Train Shape (PCs): {X_train_pca.shape}")
print(f"Test Shape (PCs): {X_test_pca.shape}")

Train Shape (PCs): (108, 2)
Test Shape (PCs): (27, 2)


In [14]:
# Define the clusters based on the K-Means clusters + PCA
leaders_list = [
    'Austria', 'Belgium', 'Czech Republic', 'Denmark', 'Finland',
    'France', 'Germany', 'Hungary', 'Italy', 'Netherlands',
    'Poland', 'Portugal', 'Romania', 'Spain', 'Sweden'
]

followers_list = [
    'Bulgaria', 'Croatia', 'Cyprus', 'Estonia', 'Greece',
    'Ireland', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta',
    'Slovakia', 'Slovenia'
]

In [15]:
# Filter Training Data 
X_train_leaders = X_train_pca[pre_covid_fixed['Country'].isin(leaders_list)]
y_train_leaders = y_train[pre_covid_fixed['Country'].isin(leaders_list)]

X_train_followers = X_train_pca[pre_covid_fixed['Country'].isin(followers_list)]
y_train_followers = y_train[pre_covid_fixed['Country'].isin(followers_list)]

# Filter Testing Data 
X_test_leaders = X_test_pca[post_covid_fixed['Country'].isin(leaders_list)]
y_test_leaders = y_test[post_covid_fixed['Country'].isin(leaders_list)]

X_test_followers = X_test_pca[post_covid_fixed['Country'].isin(followers_list)]
y_test_followers = y_test[post_covid_fixed['Country'].isin(followers_list)]

## Neural Network Models

**1. LSTM Model**

In [16]:
def run_lstm_model(X_train, y_train, X_test, y_test, cluster_name):
    # Scaling
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()

    y_train_arr = y_train.values.reshape(-1, 1)

    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)
    y_train_scaled = scaler_y.fit_transform(y_train_arr)

    # Reshape to 3D
    X_train_3D = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
    X_test_3D = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

    # Build Architecture - Trial 3 adds more capacity
    model = Sequential([
        LSTM(128, activation='tanh', input_shape=(1, X_train.shape[1]), return_sequences=True),
        Dropout(0.1),
        LSTM(64, activation='tanh', return_sequences=False),
        Dropout(0.1),
        Dense(32, activation='relu'),
        Dense(1)
    ])

    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

    # Callbacks 
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=25,
        restore_best_weights=True
    )

    # Reduces learning rate when the model stops improving to "fine-tune" weights
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=0.00001)

    # Fit the model
    model.fit(
        X_train_3D, y_train_scaled,
        epochs=500, # Increased epochs for deeper architecture
        batch_size=4,
        validation_split=0.1, # More data for training
        callbacks=[early_stop, reduce_lr],
        verbose=0
    )

    # Predict and Inverse Scale
    train_pred_scaled = model.predict(X_train_3D)
    test_pred_scaled = model.predict(X_test_3D)

    train_pred_log = scaler_y.inverse_transform(train_pred_scaled).flatten()
    test_pred_log = scaler_y.inverse_transform(test_pred_scaled).flatten()

    # Back-transform 
    y_train_real = np.exp(y_train)
    y_test_real = np.exp(y_test)
    train_pred_real = np.exp(train_pred_log)
    test_pred_real = np.exp(test_pred_log)

    # Compute Metrics
    r2_real = r2_score(y_test_real, test_pred_real)
    mae_real = mean_absolute_error(y_test_real, test_pred_real)

    print(f"{cluster_name} - LSTM Trial 3")
    print(f"Test R² (Real): {r2_real:.4f}, MAE (Real): {mae_real:.2f}%\n")

    return {
        'y_test_real': y_test_real,
        'pred_test_real': test_pred_real
    }

In [17]:
# Run for both clusters
results_lstm_leaders = run_lstm_model(X_train_leaders, y_train_leaders, X_test_leaders, y_test_leaders, "Leaders")
results_lstm_followers = run_lstm_model(X_train_followers, y_train_followers, X_test_followers, y_test_followers, "Followers")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 279ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
Leaders - LSTM Trial 3
Test R² (Real): -0.1147, MAE (Real): 11.56%



/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 809ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
Followers - LSTM Trial 3
Test R² (Real): -0.2491, MAE (Real): 6.82%



In [18]:
# Combine predictions and actual values
combined_preds = np.concatenate([
    results_lstm_leaders['pred_test_real'],
    results_lstm_followers['pred_test_real']
])

combined_actuals = pd.concat([
    results_lstm_leaders['y_test_real'],
    results_lstm_followers['y_test_real']
])

# Combine metadata in the same order (ensuring X_test slice matches post_covid_fixed order)
combined_metadata = pd.concat([
    post_covid_fixed[post_covid_fixed['Country'].isin(leaders_list)],
    post_covid_fixed[post_covid_fixed['Country'].isin(followers_list)]
])

# Build final results table
all_lstm_results = pd.DataFrame({
    'Country': combined_metadata['Country'].values,
    'Year': combined_metadata['Year'].values,
    'Actual_BEV_%': combined_actuals.values,
    'Predicted_BEV_%': combined_preds
})

# Absolute error calculation
all_lstm_results['Error_Abs'] = (all_lstm_results['Actual_BEV_%'] - all_lstm_results['Predicted_BEV_%']).abs()

# Print results by year and cluster
for year in sorted(all_lstm_results['Year'].unique()):
    print(f"\n---> LSTM: {int(year)} PREDICTION RESULTS")

    year_data = all_lstm_results[all_lstm_results['Year'] == year]

    # Leaders Display
    leaders_data = (
        year_data[year_data['Country'].isin(leaders_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )
    print(f"\n**** LEADERS CLUSTER ({int(year)}) ****")
    print(leaders_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']].to_string(index=False))

    # Followers Display
    followers_data = (
        year_data[year_data['Country'].isin(followers_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )
    print(f"\n**** FOLLOWERS CLUSTER ({int(year)}) ****")
    print(followers_data[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']].to_string(index=False))


---> LSTM: 2024 PREDICTION RESULTS

**** LEADERS CLUSTER (2024) ****
       Country  Actual_BEV_%  Predicted_BEV_%  Error_Abs
       Denmark         52.72        14.955746  37.764254
        Sweden         35.99        13.750774  22.239226
   Netherlands         35.45        15.284319  20.165681
       Finland         30.53        17.018131  13.511869
       Belgium         29.38        16.177561  13.202439
      Portugal         21.06        18.748232   2.311768
       Austria         18.58        17.029581   1.550419
        France         17.92        14.103327   3.816673
       Germany         14.47        13.447045   1.022955
       Hungary          8.04        13.431652   5.391652
       Romania          7.49        18.713444  11.223444
         Spain          6.79        16.820404  10.030404
Czech Republic          5.70        15.451037   9.751037
         Italy          5.25        15.232933   9.982933
        Poland          4.00        15.466403  11.466403

**** FOLLOWERS CL

In [19]:
# Save the results in an excel file

# Filter the results as per the clusters
all_lstm_results['Cluster'] = all_lstm_results['Country'].apply(
    lambda x: 'Leaders' if x in leaders_list else 'Followers'
)

# Create the Leaders and Followers dataFrame and sort in alphabetical order for easier visualisation
leaders_df = (
    all_lstm_results[all_lstm_results['Cluster'] == 'Leaders']
    .sort_values(by=['Country', 'Year'])
)

followers_df = (
    all_lstm_results[all_lstm_results['Cluster'] == 'Followers']
    .sort_values(by=['Country', 'Year'])
)

# Save the data frame in an excel file
leaders_df.to_excel("Trial3_LSTM_Leaders_Results.xlsx", index=False)
followers_df.to_excel("Trial3_LSTM_Followers_Results.xlsx", index=False)